# 4차시. 데이터 타입 변환과 문자열 전처리

## 1. 4차시 학습목표
타입 점검·변환, 문자열 정리, 숫자형·범주형 정제까지 이번 차시 목표를 확인합니다.

## 2. 선수지식 — 타입 확인
판다스를 불러오고 tips 데이터의 열별 타입을 확인합니다.

In [1]:
import pandas as pd                          # 판다스 불러오기

tips = pd.read_csv("data/raw/tips.csv")      # tips 데이터 불러오기
print(tips.dtypes)                           # 열별 타입만 보기

total_bill    float64
tip           float64
sex               str
smoker            str
day               str
time              str
size            int64
dtype: object


## 3. 타입별로 열 고르기 — select_dtypes
tips에서 숫자형 열과 문자열 열을 각각 골라 열 이름을 확인합니다.

In [3]:
num_df = tips.select_dtypes(include="number")
num_df

,total_bill,tip,size
0,16.99,1.01,2
1,10.34,1.66,3
2,21.01,3.50,3
3,23.68,3.31,2
4,24.59,3.61,4
...,...,...,...
239,29.03,5.92,3
240,27.18,2.00,2
241,22.67,2.00,2
242,17.82,1.75,2


In [4]:
str_df = tips.select_dtypes(include="str")
str_df

,sex,smoker,day,time
0,Female,No,Sun,Dinner
1,Male,No,Sun,Dinner
2,Male,No,Sun,Dinner
3,Male,No,Sun,Dinner
4,Female,No,Sun,Dinner
...,...,...,...,...
239,Male,No,Sat,Dinner
240,Female,Yes,Sat,Dinner
241,Male,Yes,Sat,Dinner
242,Male,No,Sat,Dinner


## 4. 지저분한 주문 데이터 — orders_raw.csv
연습용 카페 주문 데이터를 만들어 data/raw에 저장한 뒤, 다시 불러와 상태를 확인합니다.

In [5]:
orders = pd.DataFrame({
    "order_id": ["A-101", "A-102", "B-103", "A-104", "B-105", "C-106", "B-107", "A-108"],
    "product": [" 아메리카노", "카페라떼 ", "아메리카노", "바닐라라떼",
                "ICED 아메리카노", "카페라떼", "아메리카노 ", "바닐라라떼"],
    "size": ["Tall", "grande", "TALL", "Grande", " venti", "Tall", "grande", "Venti"],
    "price": ["4,500원", "5,000원", "4500", "5,500원", "5,000원", "-", "4,500원", "6,000원"],
    "qty": ["2", "1", " 3", "1", "2", "1", "삼", "2"],
    "city": ["서울", "서울시", " 부산", "부산광역시", "서울 ", "대구", "서울특별시", "부산"],
    "memo": ["쿠폰 10%", "", "포인트 500", "쿠폰 20%", "", "포인트 1000", "", "쿠폰 10%"],
})
orders.to_csv("data/raw/orders_raw.csv", index=False)   # 원본 파일로 저장

In [7]:
order_df = pd.read_csv("data/raw/orders_raw.csv")
order_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   order_id  8 non-null      str  
 1   product   8 non-null      str  
 2   size      8 non-null      str  
 3   price     8 non-null      str  
 4   qty       8 non-null      str  
 5   city      8 non-null      str  
 6   memo      5 non-null      str  
dtypes: str(7)
memory usage: 580.0 bytes


## 5. 타입 변환 — astype
문자열·정수 값을 astype으로 변환해 보고, 변환할 수 없는 값이 있으면 에러가 나는 것을 확인합니다.

In [ ]:
a = pd.Series( ["2" , "1" , "3"] )
b = a.astype(int)
b

0    2
1    1
2    3
dtype: int64

In [14]:
# order_df["qty"].astype(int)

## 6. 안전한 숫자 변환 — to_numeric
qty 열을 to_numeric으로 변환하고 변환에 실패한 건수를 확인합니다.

In [17]:
qty_s = pd.to_numeric( order_df["qty"] , errors="coerce")
qty_s

0    2.0
1    1.0
2    3.0
3    1.0
4    2.0
5    1.0
6    NaN
7    2.0
Name: qty, dtype: float64

## 7. str 접근자 — 공백·대소문자·길이
상품명 앞뒤 공백을 지우고, 사이즈를 소문자로 통일하고, 상품명 글자 수를 셉니다.

In [24]:
# 상품명 공백 제거
product = order_df["product"]
product_strip = product.str.strip()

# 사이즈 소문자 통일
size_lower = order_df["size"].str.lower()

# 상품명 글자 수
product_strip.str.len()

0     5
1     4
2     5
3     5
4    10
5     4
6     5
7     5
Name: product, dtype: int64

## 8. 문자 치환 — str.replace
가격에서 쉼표와 원을 지운 뒤 숫자로 변환합니다.

In [32]:
p = order_df["price"]
p = p.str.replace("," , "")
p = p.str.replace("원" , "")
p = pd.to_numeric(p , errors="coerce")
p

0    4500.0
1    5000.0
2    4500.0
3    5500.0
4    5000.0
5       NaN
6    4500.0
7    6000.0
Name: price, dtype: float64

## 9. 문자열 조건 — contains · startswith
라떼가 들어간 주문과 A로 시작하는 주문번호를 문자열 조건으로 찾습니다.

In [ ]:
latte = order_df["product"].str.contains("라떼")
order_a = order_df["order_id"].str.startswith("A")

0     True
1     True
2    False
3     True
4    False
5    False
6    False
7     True
Name: order_id, dtype: bool

## 10. 나누기 — str.split
주문번호를 하이픈 기준으로 나눠 지점 코드와 주문 번호 열로 펼칩니다.

In [40]:
result = order_df["order_id"].str.split("-" , expand=True)
result.columns = ["aaa" , "bbb"]
result

,aaa,bbb
0,A,101
1,A,102
2,B,103
3,A,104
4,B,105
5,C,106
6,B,107
7,A,108


## 11. 추출 — str.extract
메모에서 숫자 부분만 정규식으로 꺼냅니다.

In [ ]:
amount = order_df["memo"].str.extract(r"(\d+)")   # 숫자 부분만 추출
print(amount)

      0
0    10
1   NaN
2   500
3    20
4   NaN
5  1000
6   NaN
7    10


## 12. 숫자형 정제 — qty · price를 계산 가능한 숫자로
qty와 price를 정수로 정제하고 결제 금액 amount를 계산합니다.

In [ ]:
order_df_copy = order_df.copy()

# 수량에서 공백 제거
order_df_copy["qty"] = order_df_copy["qty"].str.strip()

# 삼 -> 3
order_df_copy["qty"] = order_df_copy["qty"].str.replace("삼" , "3")

# 타입변환
order_df_copy["qty"] = pd.to_numeric(order_df_copy["qty"] , errors="coerce")

# 가격 중 "," 제거
order_df_copy["price"] = order_df_copy["price"].str.replace("," , "")

# 가격 중 "원" 제거
order_df_copy["price"] = order_df_copy["price"].str.replace("원" , "")

# str -> int 인 줄 알았으나 아직 float ...
order_df_copy["price"] = pd.to_numeric(order_df_copy["price"] , errors="coerce")

# 가격 중 결측치 중앙값으로
price_median = order_df_copy["price"].median()
order_df_copy["price"] = order_df_copy["price"].fillna(price_median)

# float -> int
order_df_copy["price"] = order_df_copy["price"].astype(int)


# 파생칼럼 추가 (총 결제금액)
order_df_copy["amount"] = order_df_copy["price"] * order_df_copy["qty"]

order_df_copy




@@@
  order_id     product    size price  qty   city      memo
0    A-101       아메리카노    Tall  4500    2     서울    쿠폰 10%
1    A-102       카페라떼   grande  5000    1    서울시       NaN
2    B-103       아메리카노    TALL  4500    3     부산   포인트 500
3    A-104       바닐라라떼  Grande  5500    1  부산광역시    쿠폰 20%
4    B-105  ICED 아메리카노   venti  5000    2    서울        NaN
5    C-106        카페라떼    Tall   123    1     대구  포인트 1000
6    B-107      아메리카노   grande  4500    3  서울특별시       NaN
7    A-108       바닐라라떼   Venti  6000    2     부산    쿠폰 10%
@@@
<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   order_id  8 non-null      str  
 1   product   8 non-null      str  
 2   size      8 non-null      str  
 3   price     8 non-null      int64
 4   qty       8 non-null      int64
 5   city      8 non-null      str  
 6   memo      5 non-null      str  
dtypes: int64(2), str(5)
memory usage: 

,order_id,product,size,price,qty,city,memo,amount
0,A-101,아메리카노,Tall,4500,2,서울,쿠폰 10%,9000
1,A-102,카페라떼,grande,5000,1,서울시,NaN,5000
2,B-103,아메리카노,TALL,4500,3,부산,포인트 500,13500
3,A-104,바닐라라떼,Grande,5500,1,부산광역시,쿠폰 20%,5500
4,B-105,ICED 아메리카노,venti,5000,2,서울,NaN,10000
5,C-106,카페라떼,Tall,123,1,대구,포인트 1000,123
6,B-107,아메리카노,grande,4500,3,서울특별시,NaN,13500
7,A-108,바닐라라떼,Venti,6000,2,부산,쿠폰 10%,12000


## 13. 범주형 정제 — 표기 통일
도시 공백을 지우고 표준 표기 사전으로 서울·부산 표기를 통일합니다.

In [ ]:
order_df_copy["city"] = order_df_copy["city"].str.strip()

d = {
    "서울시" : "서울" ,
    "서울특별시" : "서울" ,
    "부산광역시" : "부산" ,
}
order_df_copy["city"] = order_df_copy["city"].str.replace( d )

order_df_copy["city"].value_counts()

city
서울    4
부산    3
대구    1
Name: count, dtype: int64

## 14. 범주형 타입 — category
통일한 도시 열을 category 타입으로 바꾸고 범주 목록을 확인합니다.

In [76]:
city_cate = order_df_copy["city"].astype("category")
city_cate.cat.categories

Index(['대구', '부산', '서울'], dtype='str')

## 15. 순서 있는 범주 — pd.Categorical
tips의 요일을 실제 요일 순서가 있는 범주로 바꾸고, 그 순서대로 개수와 주말 건수를 확인합니다.

In [84]:
day_order = ["Thur" , "Fri" , "Sat" , "Sun"]
tips["day"] = pd.Categorical(
    tips["day"] ,
    ordered=True ,
    categories=day_order,
)

weekend = tips["day"] >= "Sat"
weekend

0       True
1       True
2       True
3       True
4       True
       ...  
239     True
240     True
241     True
242     True
243    False
Name: day, Length: 244, dtype: bool

## 16. 종합 실습 — clean_orders 함수
주문 데이터 정제 과정을 함수 하나로 묶고, 정제 결과와 최종 타입을 확인합니다.

In [103]:
def clean_orders(path):

    df = pd.read_csv(path)

    # 문자열 공백, 대소문자 정리
    df["product"] = df["product"].str.strip()
    df["size"] = df["size"].str.strip()
    df["size"] = df["size"].str.lower()
    df["city"] = df["city"].str.strip()
    df["price"] = df["price"].str.strip()
    df["qty"] = df["qty"].str.strip()
    
    # 숫자형 정제
    df["qty"] = df["qty"].str.replace({"삼":"3"})
    df["qty"] = pd.to_numeric(df["qty"] , errors="coerce")
    df["price"] = df["price"].str.replace({",":"" , "원":""})
    df["price"] = pd.to_numeric(df["price"] , errors="coerce")
    price_median = df["price"].median()
    df["price"] = df["price"].fillna(price_median)
    df["price"] = df["price"].astype(int)
    
    # 범주형 정제
    city_map = {
        "서울시" : "서울" ,
        "서울특별시" : "서울" ,
        "부산광역시" : "부산" ,
    }
    df["city"] = df["city"].replace(city_map)
    df["city"] = df["city"].astype("category")
    df["size"] = pd.Categorical(
        df["size"] , 
        categories=["small" , "tall" , "grande" , "venti" , "big"],
        ordered=True,
    )

    # 파생 정보 (지점:branch , 결제금액:amount)
    order_id_df = df["order_id"].str.split("-" , expand=True)
    df["branch"] = order_id_df[0]
    df["amount"] = df["price"] * df["qty"]

    return df

clean_df = clean_orders("data/raw/orders_raw.csv")
clean_df

,order_id,product,size,price,qty,city,memo,branch,amount
0,A-101,아메리카노,tall,4500,2,서울,쿠폰 10%,A,9000
1,A-102,카페라떼,grande,5000,1,서울,NaN,A,5000
2,B-103,아메리카노,tall,4500,3,부산,포인트 500,B,13500
3,A-104,바닐라라떼,grande,5500,1,부산,쿠폰 20%,A,5500
4,B-105,ICED 아메리카노,venti,5000,2,서울,NaN,B,10000
5,C-106,카페라떼,tall,5000,1,대구,포인트 1000,C,5000
6,B-107,아메리카노,grande,4500,3,서울,NaN,B,13500
7,A-108,바닐라라떼,venti,6000,2,부산,쿠폰 10%,A,12000


## 17. 정제 결과 저장과 활용
정제 결과를 data/processed에 저장하고 총 매출과 도시별 건수를 확인합니다.

In [104]:
clean_df.to_csv("data/processed/orders_clean.csv" , index=False)
print("저장완료 ~~~")

저장완료 ~~~
